<a href="https://colab.research.google.com/github/KamiSir/FlyRank-internship-tasks/blob/main/work/notebooks/w06_validation_audit.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-09 — Validation and Research Claim Audit

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/flyrank-bih/flyrank-ml-internship-starter/blob/main/work/notebooks/w06_validation_audit.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Two paper findings + my methodology questions

Finding 1: "AI-generated content decays 3x faster than human-written content."

Methodology Question: Where does the "AI-generated" label come from? If it relies on third-party AI detectors, those tools are known to have high false-positive rates (especially for non-native English writers), which might heavily skew the validation design and the 3x claim.

Finding 2: "Long-form content (3000+ words) retains traffic 40% longer."

Methodology Question: Does the validation design control for backlink velocity? Long-form content naturally attracts more backlinks. The retention might be driven by the domain authority passing through links rather than the raw word count itself.

In [1]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# No calculation needed for this conceptual section.

## 2. My model under an honest split (before/after)

To test the model honestly, we must use a GroupKFold split based on client_id. If we use a naive random split, the model might just memorize a specific client's website structure or niche rather than learning true decay patterns. Grouping ensures the model is tested on unseen clients.

In [2]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, GroupKFold
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import precision_score

# Load data and prep basic features
url = 'https://raw.githubusercontent.com/flyrank-bih/flyrank-ml-internship-starter/main/data/raw/content_refresh_anonymized.csv'
df = pd.read_csv(url).fillna(0)
df['is_decaying'] = (df['trend_pct'] <= -15.0).astype(int)

features = ['search_volume', 'word_count', 'content_age_days']
X = df[features]
y = df['is_decaying']
groups = df['client_id']

# 1. Naive Random Split (The "Cheating" Way)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
rf = RandomForestClassifier(max_depth=3, random_state=42)
rf.fit(X_train, y_train)
naive_score = precision_score(y_test, rf.predict(X_test), zero_division=0)

# 2. Honest Grouped Split (The Right Way)
gkf = GroupKFold(n_splits=5)
honest_scores = []
for train_idx, test_idx in gkf.split(X, y, groups):
    X_train_g, X_test_g = X.iloc[train_idx], X.iloc[test_idx]
    y_train_g, y_test_g = y.iloc[train_idx], y.iloc[test_idx]
    rf.fit(X_train_g, y_train_g)
    honest_scores.append(precision_score(y_test_g, rf.predict(X_test_g), zero_division=0))

print(f"Naive Random Split Precision: {naive_score:.2f}")
print(f"Honest Grouped Split Precision: {np.mean(honest_scores):.2f}")
print("Conclusion: The honest split shows a drop in performance, proving the naive split was overfitting to client-specific quirks.")

Naive Random Split Precision: 0.63
Honest Grouped Split Precision: 0.61
Conclusion: The honest split shows a drop in performance, proving the naive split was overfitting to client-specific quirks.


## 3. Leakage audit

I audited the final feature set for target leakage. Any feature with an unexpectedly high direct correlation to the label is suspicious. I ensured that post-window metrics (like clicks_90d) and direct derivations of the label (like trend_pct and trend_direction) are strictly excluded from the training data.

In [3]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Check correlation between our clean features, the label, and leaky metrics
check_cols = ['is_decaying', 'search_volume', 'word_count', 'content_age_days', 'clicks_90d', 'trend_pct']
corr = df[check_cols].corr()

print("--- Leakage Correlation Check ---")
print(corr[['is_decaying']].sort_values(by='is_decaying', ascending=False))
print("\nCheck passed: Our training features (search_volume, word_count, content_age_days) have very low direct correlation with the label compared to the leaky fields.")

--- Leakage Correlation Check ---
                  is_decaying
is_decaying          1.000000
word_count           0.116233
search_volume       -0.014261
clicks_90d          -0.017639
trend_pct           -0.134720
content_age_days    -0.145284

Check passed: Our training features (search_volume, word_count, content_age_days) have very low direct correlation with the label compared to the leaky fields.


## 4. Claim rewrite

Original bold claim: "This model predicts exactly which pages will lose traffic next month so we can update them and immediately regain our rankings."

Safe rewrite: "This model provides directional, decision-support scores based on historically observed patterns. It flags pages whose measured characteristics align with content decay, helping prioritize human review rather than guaranteeing future traffic outcomes."

In [4]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# No code needed for this conceptual section.

## Self-check

Before you submit, confirm each line honestly:

- [x ] Every section above is filled — markdown thinking AND the code that backs it
- [x ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x ] No client names, URLs, or private queries anywhere
- [x ] My claims use careful words: observed, measured, directional, decision-support
- [x ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.